# Results

Reads the [results store](../autobus-docs/docs/concepts/experiment.md) and
nothing else. **No simulation runs here**, and no CARLA server is needed — so
this works whether or not an experiment is in progress, and shows everything
recorded so far.

Re-run it any time. If a campaign is running in another process, re-running
this simply picks up the trials that have finished since.

Each chart sits beside the table it summarises. That is partly for reading and
partly a requirement: one palette slot falls below 3:1 contrast on a light
surface, and a visible table is the accepted relief.

## House style

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler

# Categorical slots 1-3 of the project's validated palette. Checked rather
# than chosen by eye: all-pairs CVD separation 9.2 (>=8), normal-vision 24.0
# (>=15), both on the light surface below. Three is the cap that clears
# all-pairs; a fourth series folds into "Other" or gets its own facet.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
PRIMARY = SERIES[0]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE,
    "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT,
    "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.labelsize": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "figure.dpi": 110,
})
pd.set_option("display.max_rows", 200)
print("style applied")

In [ ]:
import math
import statistics

from scipy import stats

from src.experiment.store import DEFAULT_RUNS_DIR, load_frame, load_runs
from src.experiment.summary import summarize_results
from src.experiment.store import to_results

from IPython.display import Markdown, display

RUNS = DEFAULT_RUNS_DIR
frame = load_frame(RUNS)
records = load_runs(RUNS)

if frame.empty:
    print("The store is empty. Run notebooks/synthetic-campaign.ipynb first.")
else:
    print(f"{len(frame)} runs recorded")

## What is in the store

A run is one `(parameters, seed)` pair. Runs from different maps, backends,
loads or policy settings are different runs and never merge, so everything
recorded — synthetic and CARLA alike — coexists here.

In [ ]:
KEY = [
    "identity.world.backend",
    "identity.world.town" if "identity.world.town" in frame.columns
    else "identity.world.map",
    "identity.trial.max_requests",
    "identity.trial.demand_interval_s",
    "identity.trial.max_wait_s",
]
KEY = [c for c in KEY if c in frame.columns]


def scenarios(f):
    """
    Group runs by the setting they share, ignoring policy and seed.
    :param f: The store as a frame.
    :return: One row per scenario, with its run and seed counts.
    """
    grouped = f.groupby(KEY, dropna=False).agg(
        runs=("run_id", "count"),
        policies=("identity.policy", "nunique"),
        seeds=("identity.seed", "nunique"),
    )
    return grouped.sort_values("runs", ascending=False)


if not frame.empty:
    display(scenarios(frame))

## Pick a scenario

Defaults to the one with the most runs. Change `SCENARIO` to any row index
from the table above to look at a different one.

In [ ]:
SCENARIO = None   # None = the scenario with the most runs

if not frame.empty:
    table = scenarios(frame)
    chosen = table.index[0] if SCENARIO is None else SCENARIO
    if not isinstance(chosen, tuple):
        chosen = (chosen,)
    mask = pd.Series(True, index=frame.index)
    for column, value in zip(KEY, chosen):
        mask &= (frame[column].isna() if pd.isna(value)
                 else frame[column] == value)
    subset = frame[mask]
    wanted = set(subset["run_id"])
    scenario_records = [r for r in records if r.run_id in wanted]
    print(dict(zip(KEY, chosen)))
    print(f"{len(scenario_records)} runs, "
          f"{subset['identity.policy'].nunique()} policies, "
          f"{subset['identity.seed'].nunique()} seeds")

## How the policies compare

In [ ]:
if not frame.empty:
    results = to_results(scenario_records, paired=True)
    display(Markdown(summarize_results(results).markdown))

The bars below show the same numbers with 95% confidence intervals. One
series per panel, so the category axis carries identity and no legend is
needed; a shorter bar is better for wait and ride, longer for the other two.

In [ ]:
KPIS = [
    ("served_fraction", "Served fraction", 100.0, "%", "higher"),
    ("mean_wait_time", "Mean wait", 1.0, "s", "lower"),
    ("mean_ride_time", "Mean ride", 1.0, "s", "lower"),
    ("throughput", "Throughput", 1.0, "req/min", "higher"),
]


def kpi_stats(results, field, scale):
    """
    Mean and 95% CI half-width per policy for one KPI.
    :param results: Policy label to trial results.
    :param field: KPISnapshot attribute name.
    :param scale: Multiplier applied before reporting.
    :return: Mapping of label to (mean, half-width).
    """
    out = {}
    for label, runs in results.items():
        values = [getattr(r.kpi, field) for r in runs
                  if getattr(r.kpi, field) is not None]
        if not values:
            continue
        mean = statistics.fmean(values) * scale
        if len(values) > 1:
            sd = statistics.stdev(values) * scale
            half = stats.t.ppf(0.975, len(values) - 1) * sd / len(values) ** .5
        else:
            half = 0.0
        out[label] = (mean, half)
    return out


def kpi_panel(results, kpis=KPIS):
    """
    Draw one horizontal bar panel per KPI, sorted best-first.
    :param results: Policy label to trial results.
    :param kpis: (field, title, scale, unit, direction) tuples.
    :return: The figure.
    """
    figure, axes = plt.subplots(2, 2, figsize=(11, 1.0 + 0.42 * len(results)
                                               * 2))
    for axis, (field, title, scale, unit, direction) in zip(
            axes.ravel(), kpis):
        data = kpi_stats(results, field, scale)
        fmt = (lambda v: f"{v:.3f}") if unit == "req/min" else (
            (lambda v: f"{v:.1f}%") if unit == "%" else
            (lambda v: f"{v:,.0f} s"))
        order = sorted(data, key=lambda k: data[k][0],
                       reverse=(direction == "higher"))
        values = [data[k][0] for k in order]
        errors = [data[k][1] for k in order]
        positions = range(len(order))
        axis.barh(positions, values, xerr=errors, color=PRIMARY, height=0.62,
                  error_kw=dict(ecolor=INK_SOFT, lw=1.0, capsize=2))
        axis.set_yticks(list(positions), order)
        axis.invert_yaxis()
        axis.set_title(f"{title}  ({'higher' if direction == 'higher' else 'lower'} is better)")
        axis.set_xlabel(unit)
        axis.grid(axis="x")
        axis.set_axisbelow(True)
        axis.spines["left"].set_visible(False)
        axis.tick_params(axis="y", length=0)
        # Labels sit beyond the whisker, not on top of it: the error bar is
        # the reason a reader trusts the bar, so it must stay legible.
        reach = max((v + e for v, e in zip(values, errors)), default=1.0)
        for position, value, error in zip(positions, values, errors):
            axis.text(value + error + reach * 0.025, position,
                      fmt(value), va="center", fontsize=8, color=INK_SOFT)
        axis.set_xlim(0, reach * 1.26)
    figure.tight_layout()
    return figure


if not frame.empty:
    kpi_panel(results)
    plt.show()

## Against the design review's criteria

Two things are asked for: fulfilment at or above 80%, and a **positive sum of
percentage differences** across the other three KPIs when compared with a
fixed-route bus. Both are computed here against whichever fixed-route policy
is present in the scenario.

In [ ]:
def criteria(results, baseline_hint="Fixed route"):
    """
    Score every policy against the fixed-route baseline.
    :param results: Policy label to trial results.
    :param baseline_hint: Label of the comparator.
    :return: Markdown table, or a note if no baseline is present.
    """
    if baseline_hint not in results:
        return f"*No `{baseline_hint}` runs in this scenario to compare with.*"

    def mean_of(runs, field):
        """Average one KPI over a policy's runs."""
        values = [getattr(r.kpi, field) for r in runs
                  if getattr(r.kpi, field) is not None]
        return statistics.fmean(values) if values else float("nan")

    base = results[baseline_hint]
    rows = ["| policy | fulfilment | wait | ride | throughput | sum | verdict |",
            "| --- | --- | --- | --- | --- | --- | --- |"]
    for label, runs in sorted(results.items()):
        if label == baseline_hint:
            continue
        fulfilment = mean_of(runs, "fulfillment_rate") * 100
        wait = (mean_of(base, "mean_wait_time")
                - mean_of(runs, "mean_wait_time")) / mean_of(
                    base, "mean_wait_time")
        ride = (mean_of(base, "mean_ride_time")
                - mean_of(runs, "mean_ride_time")) / mean_of(
                    base, "mean_ride_time")
        thru = (mean_of(runs, "throughput")
                - mean_of(base, "throughput")) / mean_of(base, "throughput")
        total = (wait + ride + thru) * 100
        passes = fulfilment >= 80 and total > 0
        rows.append(
            f"| {label} | {fulfilment:.1f}% | {wait * 100:+.1f}% | "
            f"{ride * 100:+.1f}% | {thru * 100:+.1f}% | {total:+.1f}% | "
            f"{'**PASS**' if passes else 'fail'} |")
    return "\n".join(rows)


if not frame.empty:
    display(Markdown(criteria(results)))

## Is the sample big enough?

The bar is a confidence-interval half-width below **5% of the mean** on every
KPI. Where it is missed, the last column says how many seeds would reach it —
and extending a campaign only runs what is missing.

In [ ]:
def precision(results, confidence=0.95):
    """
    Report achieved CI half-width per policy and KPI.
    :param results: Policy label to trial results.
    :param confidence: Confidence level.
    :return: Markdown table.
    """
    rows = ["| policy | KPI | mean | CI half | % of mean | seeds for 5% |",
            "| --- | --- | --- | --- | --- | --- |"]
    for label, runs in sorted(results.items()):
        for field, title, scale, unit, _ in KPIS:
            values = [getattr(r.kpi, field) for r in runs
                      if getattr(r.kpi, field) is not None]
            n = len(values)
            if n < 2:
                continue
            mean = statistics.fmean(values) * scale
            sd = statistics.stdev(values) * scale
            if mean == 0 or sd == 0:
                rows.append(f"| {label} | {title} | {mean:.4g}{unit} | 0 | "
                            f"0.0% | met |")
                continue
            half = stats.t.ppf(1 - (1 - confidence) / 2, n - 1) * sd / n ** .5
            pct = 100 * half / abs(mean)
            need = "met" if pct <= 5 else str(math.ceil(n * (pct / 5) ** 2))
            rows.append(f"| {label} | {title} | {mean:.4g}{unit} | "
                        f"{half:.3g} | {pct:.1f}% | {need} |")
    return "\n".join(rows)


if not frame.empty:
    display(Markdown(precision(results)))

## Where the time actually goes

Averages hide the tail. These are every fulfilled passenger's wait, pooled
across seeds — the box spans the quartiles, the whiskers the 5th–95th
percentile, and the notch is the median.

In [ ]:
def wait_spread(results, field="wait_times", title="Wait time"):
    """
    Box plot of a per-passenger distribution across policies.
    :param results: Policy label to trial results.
    :param field: TrialResult attribute holding the samples.
    :param title: Chart title.
    :return: The figure, or None when there is nothing to draw.
    """
    pooled = {
        label: [v for r in runs for v in getattr(r, field)]
        for label, runs in results.items()
    }
    pooled = {k: v for k, v in pooled.items() if v}
    if not pooled:
        return None
    order = sorted(pooled, key=lambda k: statistics.median(pooled[k]))
    figure, axis = plt.subplots(figsize=(9, 0.44 * len(order) + 1.4))
    axis.boxplot([pooled[k] for k in order],
                 orientation="horizontal", notch=True,
                 whis=(5, 95), showfliers=False, widths=0.6,
                 patch_artist=True,
                 boxprops=dict(facecolor=PRIMARY, edgecolor=PRIMARY,
                               alpha=0.85),
                 medianprops=dict(color=SURFACE, lw=1.6),
                 whiskerprops=dict(color=INK_SOFT),
                 capprops=dict(color=INK_SOFT))
    axis.set_yticks(range(1, len(order) + 1), order)
    axis.set_xlabel("seconds")
    axis.set_title(f"{title} per passenger (lower is better)")
    axis.grid(axis="x")
    axis.set_axisbelow(True)
    axis.spines["left"].set_visible(False)
    axis.tick_params(axis="y", length=0)
    figure.tight_layout()
    return figure


if not frame.empty:
    wait_spread(results)
    plt.show()
    wait_spread(results, "ride_times", "Ride time")
    plt.show()

## The service window

Only present once a window sweep has run. One line per policy family, with a
95% band across seeds; the x axis is the window the policy was told to keep.

In [ ]:
WINDOW_COLUMN = "identity.policy_params.max_wait_s"

if not frame.empty and WINDOW_COLUMN in subset.columns:
    swept = subset[subset[WINDOW_COLUMN].notna()].copy()
    if swept[WINDOW_COLUMN].nunique() > 1:
        swept["family"] = swept["identity.policy_class"].str.replace(
            "InsertionPolicy", "", regex=False)
        from src.experiment import plot_parameter_sweep
        for metric, title in (("kpi.mean_wait_time", "Mean wait"),
                              ("kpi.mean_ride_time", "Mean ride"),
                              ("kpi.served_fraction", "Served fraction")):
            plot_parameter_sweep(swept, x=WINDOW_COLUMN, y=metric,
                                 hue="family",
                                 title=f"{title} against service window")
            plt.show()
    else:
        print("Only one service window recorded in this scenario — "
              "run stage 2 of the campaign notebook for the sweep.")

## CARLA

The transfer demonstration: whether what the synthetic map establishes still
holds when a real vehicle drives a real town. Shown separately because a
CARLA run is a different world, not another row in the same table.

In [ ]:
carla = frame[frame["identity.world.backend"] == "CarlaBackend"] \
    if "identity.world.backend" in frame.columns else frame.iloc[0:0]

if carla.empty:
    print("No CARLA runs recorded.")
else:
    biggest = carla.groupby("identity.trial.max_requests").size().idxmax()
    chosen_carla = carla[carla["identity.trial.max_requests"] == biggest]
    wanted_carla = set(chosen_carla["run_id"])
    carla_records = [r for r in records if r.run_id in wanted_carla]
    carla_results = to_results(carla_records, paired=True)
    print(f"{len(carla_records)} runs, {biggest} requests, "
          f"{chosen_carla['identity.seed'].nunique()} seeds, "
          f"town {chosen_carla['identity.world.town'].iloc[0]}, "
          f"{chosen_carla['identity.world.npc_vehicles'].iloc[0]:.0f} NPCs")
    display(Markdown(summarize_results(carla_results).markdown))
    display(Markdown(criteria(carla_results)))
    kpi_panel(carla_results)
    plt.show()

## Was the driving healthy?

Only CARLA reports these. `repaths` is the one to watch: it counts a bus that
stopped making progress for reasons other than a red light or the vehicle in
front, so anything above zero is worth a look rather than a shrug.

In [ ]:
DIAGNOSTICS = ["arrivals", "distance_m", "repaths", "queued_ticks",
               "red_light_ticks", "walker_spawn_failures", "unroutable_legs"]

if not carla.empty:
    rows = []
    for record in carla_records:
        row = {"policy": record.identity.policy,
               "seed": record.identity.seed}
        row.update({k: record.backend_stats.get(k) for k in DIAGNOSTICS})
        rows.append(row)
    health = pd.DataFrame(rows)
    display(health.groupby("policy")[DIAGNOSTICS].mean().round(1))
    total_repaths = health["repaths"].sum()
    print(f"total re-routes across all runs: {total_repaths:.0f}"
          + ("  (clean)" if total_repaths == 0 else "  <-- investigate"))

## Provenance

Which commit produced each run, and whether its hash seed was pinned. A run
recorded with `python_hash_seed` unset cannot be reproduced exactly, because
several policies break ties by set-iteration order.

In [ ]:
if not frame.empty:
    prov = frame[["provenance.git_commit", "provenance.python_hash_seed"]] \
        .fillna("unset").value_counts().to_frame("runs")
    display(prov)
    unpinned = (frame["provenance.python_hash_seed"].isna()).sum()
    if unpinned:
        print(f"{unpinned} runs were recorded without PYTHONHASHSEED pinned; "
              f"their exact reproduction is not guaranteed.")
    else:
        print("every run was recorded with PYTHONHASHSEED pinned")